# AI와 함께 타이타닉 데이터 분석 한 사이클 완주하기

STEP 00–17 학생용 실행 Notebook입니다.

핵심 원칙은 **셀을 위에서 아래로 직접 실행하면서 데이터가 어떻게 변하는지 확인하는 것**입니다. STEP 11 이후에도 `titanic_app.features` 같은 사용자 정의 모듈에서 핵심 Feature 로직을 가져오지 않고, Notebook 안에서 직접 작성합니다.


## STEP 00. 전체 분석 지도

환경 → 데이터 → 품질 → Target → 결측/컬럼/인코딩 원리 → 시각화 → EDA/통계 → Feature → split → Pipeline → Baseline → 평가 → 추가 모델 → 최종 선택 → 저장/새 예측 → Streamlit


## STEP 01. 실행 환경 확인


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

print("Python:", sys.executable)
print("Version:", sys.version.split()[0])
print("Working directory:", Path.cwd())
print("pandas / numpy / sklearn:", pd.__version__, np.__version__, sklearn.__version__)


## STEP 02. 데이터 로딩

`data/titanic/train.csv`가 없다면 저장소 루트에서 먼저 `python scripts/prepare_titanic_data.py`를 실행합니다.


In [ ]:
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd

data_path = project_root / "data" / "titanic" / "train.csv"
if not data_path.is_file():
    raise FileNotFoundError(
        f"{data_path} 파일이 없습니다. 저장소 루트에서 "
        "python scripts/prepare_titanic_data.py 를 먼저 실행하세요."
    )

df = pd.read_csv(data_path)
display(df.head())
print("shape:", df.shape)


## STEP 03. 데이터 구조와 품질 확인


In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_rate": (df.isna().mean() * 100).round(2),
    "nunique": df.nunique(dropna=False),
})
display(quality)
display(df.describe())

for col in ["Survived", "Pclass", "Sex", "Embarked"]:
    print(f"\n[{col}]")
    display(df[col].value_counts(dropna=False).to_frame("count"))


## STEP 04. Target 정의

`Survived=1`을 Positive class로 두는 이진 분류 문제로 정의합니다.


In [ ]:
print(df["Survived"].value_counts(dropna=False).sort_index())
print({
    "task": "binary_classification",
    "target": "Survived",
    "positive_class": 1,
})


## STEP 05. 결측치 처리 원리 학습

여기서 만드는 `df_work`는 **EDA용**입니다. 모델링에서는 다시 원본 `df`에서 시작합니다.


In [ ]:
df_work = df.copy()

age_median_for_eda = df_work["Age"].median()
embarked_mode_for_eda = df_work["Embarked"].mode(dropna=True).iloc[0]

print("EDA Age median:", age_median_for_eda)
print("EDA Embarked mode:", embarked_mode_for_eda)

df_work["Age"] = df_work["Age"].fillna(age_median_for_eda)
df_work["Embarked"] = df_work["Embarked"].fillna(embarked_mode_for_eda)

display(df_work[["Age", "Embarked", "Cabin"]].isna().sum().to_frame("missing"))


## STEP 06. 컬럼 사용 정책 검토


In [ ]:
candidate_cols = ["PassengerId", "Name", "Ticket", "Cabin"]
for col in candidate_cols:
    print(f"\n[{col}] dtype={df_work[col].dtype}, "
          f"nunique={df_work[col].nunique(dropna=False)}, "
          f"missing={df_work[col].isna().sum()}")
    print(df_work[col].dropna().astype(str).head(5).tolist())


## STEP 07. 범주형 인코딩 원리 학습

`df_encoded`는 인코딩 원리를 보기 위한 연습용이며 모델 입력으로 사용하지 않습니다.


In [ ]:
df_encoded = df_work.copy()
encoded_preview = pd.get_dummies(
    df_encoded[["Sex", "Embarked"]],
    columns=["Sex", "Embarked"],
    dtype=int,
)
display(encoded_preview.head())


## STEP 08. 시각화


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(data=df_work, x="Survived")
plt.title("Survived distribution")
plt.show()

sns.barplot(data=df_work, x="Sex", y="Survived")
plt.title("Survival rate by Sex")
plt.show()

sns.barplot(data=df_work, x="Pclass", y="Survived")
plt.title("Survival rate by Pclass")
plt.show()

sns.boxplot(data=df_work, x="Survived", y="Fare")
plt.title("Fare by Survived")
plt.show()


## STEP 09. 기초 통계와 EDA


In [ ]:
sex_summary = (
    df_work.groupby("Sex", dropna=False)
    .agg(rows=("Survived", "size"), survival_rate=("Survived", "mean"))
)
pclass_summary = (
    df_work.groupby("Pclass", dropna=False)
    .agg(rows=("Survived", "size"), survival_rate=("Survived", "mean"))
)
sex_pclass_summary = (
    df_work.groupby(["Sex", "Pclass"], dropna=False)
    .agg(rows=("Survived", "size"), survival_rate=("Survived", "mean"))
)

display(sex_summary)
display(pclass_summary)
display(sex_pclass_summary)


### STEP 09-A. Fare 평균 차이 — Welch 독립표본 t-test

- H0: 생존자와 비생존자의 평균 Fare에는 차이가 없다.
- H1: 두 그룹의 평균 Fare에는 차이가 있다.
- `p-value < 0.05`이면 보통 H0를 기각합니다.
- 통계적 차이가 있다고 해서 Fare가 생존의 원인이라는 뜻은 아닙니다.


In [ ]:
from scipy.stats import ttest_ind

survived_fare = df_work.loc[df_work["Survived"] == 1, "Fare"].dropna()
not_survived_fare = df_work.loc[df_work["Survived"] == 0, "Fare"].dropna()

stat, p_value = ttest_ind(survived_fare, not_survived_fare, equal_var=False)

print("survived Fare mean:", round(survived_fare.mean(), 2), " n =", len(survived_fare))
print("not survived Fare mean:", round(not_survived_fare.mean(), 2), " n =", len(not_survived_fare))
print("t-statistic:", round(stat, 4))
print(f"p-value: {p_value:.3e}")


## STEP 10. Feature 설계

이번 기본 실행에서는 `FamilySize`, `IsAlone`을 직접 만들어 봅니다.


In [ ]:
feature_demo = df_work.copy()
feature_demo["FamilySize"] = feature_demo["SibSp"] + feature_demo["Parch"] + 1
feature_demo["IsAlone"] = (feature_demo["FamilySize"] == 1).astype(int)

display(feature_demo[["SibSp", "Parch", "FamilySize", "IsAlone", "Survived"]].head())


# Part 2. 모델링 — STEP 11–15

이제 `df_work`, `df_encoded`, `feature_demo`가 아니라 **원본 `df`에서 다시 시작**합니다.


## STEP 11. 학습/테스트 데이터 준비

외부 `titanic_app.features` 모듈을 불러오지 않고 Feature 생성부터 split까지 직접 실행합니다.


In [ ]:
# 모델링용 데이터는 원본 df에서 다시 시작
model_source = df.copy()

# STEP 10에서 선택한 결정적 Feature를 직접 생성
model_source["FamilySize"] = (
    model_source["SibSp"] + model_source["Parch"] + 1
)
model_source["IsAlone"] = (
    model_source["FamilySize"] == 1
).astype(int)

display(model_source[["SibSp", "Parch", "FamilySize", "IsAlone"]].head())


In [ ]:
# 모델에 사용할 Feature를 직접 작성
raw_input_columns = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"
]
feature_columns = raw_input_columns + ["FamilySize", "IsAlone"]

numeric_features = [
    "Age", "SibSp", "Parch", "Fare", "FamilySize"
]
categorical_features = [
    "Pclass", "Sex", "Embarked", "IsAlone"
]

X = model_source[feature_columns].copy()
y = model_source["Survived"].astype(int).copy()

print("X:", X.shape)
print("y:", y.shape)
display(X.head())


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("train:", X_train.shape, y_train.shape)
print("test :", X_test.shape, y_test.shape)

print("전체 데이터")
display(y.value_counts(normalize=True).sort_index().rename("ratio").to_frame())
print("학습 데이터")
display(y_train.value_counts(normalize=True).sort_index().rename("train_ratio").to_frame())
print("테스트 데이터")
display(y_test.value_counts(normalize=True).sort_index().rename("test_ratio").to_frame())


## STEP 12. Baseline Pipeline

Train/Test를 나눈 **뒤에** Pipeline이 Train에서 결측치 처리 기준과 인코딩 범주, 스케일 기준을 학습합니다.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

baseline_pipeline.fit(X_train, y_train)

print("train accuracy:", round(baseline_pipeline.score(X_train, y_train), 4))
print("test accuracy :", round(baseline_pipeline.score(X_test, y_test), 4))


## STEP 13. 평가와 오류 분석


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

y_pred = baseline_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1       :", round(f1, 4))
print("Confusion Matrix:")
print(cm)
print(classification_report(y_test, y_pred, labels=[0, 1], zero_division=0))

tn, fp, fn, tp = cm.ravel()
print("TN:", tn, "FP:", fp, "FN:", fn, "TP:", tp)


In [ ]:
error_df = X_test.copy()
error_df["actual"] = y_test
error_df["predicted"] = y_pred

misclassified = error_df[error_df["actual"] != error_df["predicted"]].copy()
false_positive = error_df[(error_df["actual"] == 0) & (error_df["predicted"] == 1)].copy()
false_negative = error_df[(error_df["actual"] == 1) & (error_df["predicted"] == 0)].copy()

print("misclassified:", len(misclassified))
print("FP:", len(false_positive))
print("FN:", len(false_negative))
display(misclassified.head(10))


### Confusion Matrix 빠른 해석

- 스팸 검출: FP는 정상 메일을 스팸으로 잘못 막은 경우
- 암 검진: FN은 실제 암인데 정상이라고 놓친 경우

어떤 오류가 더 중요한지는 문제의 비용에 따라 달라집니다.


## STEP 14. 추가 모델 — Random Forest


In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

additional_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestClassifier(n_estimators=300, random_state=42)),
])

additional_pipeline.fit(X_train, y_train)
additional_pred = additional_pipeline.predict(X_test)

additional_accuracy = accuracy_score(y_test, additional_pred)
additional_precision = precision_score(y_test, additional_pred, pos_label=1, zero_division=0)
additional_recall = recall_score(y_test, additional_pred, pos_label=1, zero_division=0)
additional_f1 = f1_score(y_test, additional_pred, pos_label=1, zero_division=0)
additional_cm = confusion_matrix(y_test, additional_pred, labels=[0, 1])

comparison = pd.DataFrame([
    {"model": "Logistic Regression", "accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1},
    {"model": "Random Forest", "accuracy": additional_accuracy, "precision": additional_precision, "recall": additional_recall, "f1": additional_f1},
])

display(comparison)
print("Random Forest confusion matrix")
print(additional_cm)


## STEP 15. Cross Validation과 최종 모델 선택


In [ ]:
from sklearn.model_selection import cross_validate

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
}

baseline_cv = cross_validate(
    baseline_pipeline, X_train, y_train, cv=5, scoring=scoring
)
additional_cv = cross_validate(
    additional_pipeline, X_train, y_train, cv=5, scoring=scoring
)

cv_comparison = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "cv_accuracy": baseline_cv["test_accuracy"].mean(),
        "cv_precision": baseline_cv["test_precision"].mean(),
        "cv_recall": baseline_cv["test_recall"].mean(),
        "cv_f1": baseline_cv["test_f1"].mean(),
    },
    {
        "model": "Random Forest",
        "cv_accuracy": additional_cv["test_accuracy"].mean(),
        "cv_precision": additional_cv["test_precision"].mean(),
        "cv_recall": additional_cv["test_recall"].mean(),
        "cv_f1": additional_cv["test_f1"].mean(),
    },
])

display(cv_comparison)


### 개인 판단 지점 — 최종 모델 선택

아래 기본값을 실제 비교 결과와 자신의 판단에 따라 `"baseline"` 또는 `"random_forest"`로 바꿀 수 있습니다.


In [ ]:
FINAL_MODEL_CHOICE = "baseline"

if FINAL_MODEL_CHOICE == "baseline":
    final_pipeline = baseline_pipeline
    final_model_name = "Logistic Regression"
elif FINAL_MODEL_CHOICE == "random_forest":
    final_pipeline = additional_pipeline
    final_model_name = "Random Forest"
else:
    raise ValueError("FINAL_MODEL_CHOICE는 baseline 또는 random_forest여야 합니다.")

print("final model:", final_model_name)


# Part 3. 저장·새 입력 예측·서비스


## STEP 16. 모델 저장과 신규 승객 예측


In [ ]:
import json
import joblib

models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

pipeline_path = models_dir / "titanic_final_pipeline.joblib"
contract_path = models_dir / "titanic_model_contract.json"

joblib.dump(final_pipeline, pipeline_path)

model_contract = {
    "task": "binary_classification",
    "target": "Survived",
    "positive_class": 1,
    "raw_input_columns": raw_input_columns,
    "model_feature_columns": feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "derived_features": ["FamilySize", "IsAlone"],
    "final_estimator": type(final_pipeline.named_steps["model"]).__name__,
}

contract_path.write_text(
    json.dumps(model_contract, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("saved pipeline:", pipeline_path)
print("saved contract:", contract_path)
display(model_contract)


In [ ]:
loaded_pipeline = joblib.load(pipeline_path)

new_passenger = pd.DataFrame([
    {
        "Pclass": 3,
        "Sex": "male",
        "Age": 30.0,
        "SibSp": 0,
        "Parch": 0,
        "Fare": 10.0,
        "Embarked": "S",
    }
])

# 학습 때와 같은 파생 Feature 규칙을 직접 적용
new_passenger["FamilySize"] = (
    new_passenger["SibSp"] + new_passenger["Parch"] + 1
)
new_passenger["IsAlone"] = (
    new_passenger["FamilySize"] == 1
).astype(int)

new_model_input = new_passenger[feature_columns].copy()

display(new_model_input)

new_prediction = int(loaded_pipeline.predict(new_model_input)[0])
new_probabilities = loaded_pipeline.predict_proba(new_model_input)[0]
classes = loaded_pipeline.named_steps["model"].classes_
positive_index = list(classes).index(1)
positive_probability = float(new_probabilities[positive_index])

print("prediction:", new_prediction)
print("survival probability:", round(positive_probability, 4))


## STEP 17. Streamlit 서비스

Notebook에서 직접 확인한 입력 흐름을 Streamlit으로 옮깁니다.

수업에서는 먼저 다음 흐름을 이해합니다.

`원본 입력 → FamilySize → IsAlone → feature_columns → 저장된 Pipeline.predict()`

저장소의 `src/titanic_app/app.py`는 이 반복 코드를 실무적으로 재사용하기 위해 모듈화한 배포 버전입니다. **모듈화는 원리를 이해한 뒤의 리팩터링 단계**로 봅니다.

실행:

```powershell
streamlit run src/titanic_app/app.py
```


# 최종 정리

- STEP 07 인코딩은 원리 학습용입니다.
- 실제 모델링에서는 STEP 11에서 원본 `df`로 다시 시작합니다.
- STEP 11에서 먼저 Train/Test를 나눕니다.
- STEP 12 Pipeline이 Train에서 결측치 처리·One-Hot·Scaling 기준을 학습합니다.
- STEP 13에서 Confusion Matrix와 FP/FN까지 봅니다.
- STEP 16에서는 새 승객의 파생 Feature도 직접 생성해 저장된 Pipeline으로 예측합니다.
- 원리를 이해한 뒤에만 반복 코드를 함수/모듈로 리팩터링합니다.
